In [ ]:
# ---------------- Imports ----------------
import os
import json
import sys
import textwrap

import yaml
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel



In [ ]:
# ---------------- Args ----------------
model_choice = "meta-llama/Llama-3.1-8B-Instruct"
#model_choice = "meta-llama/Llama-3.2-3B-Instruct"
#model_choice = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
adapter_choice = "20260126T2330-llama-3.1-8b-instruct-20260115T095923-combined-claims-200k-authoritative-0.25"



In [ ]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

data_path = config["paths"]["proj-store"]
models_folderpath = config["paths"]["models"]


base_model_path = os.path.join(models_folderpath, model_choice)
adapter_model_path = os.path.join(data_path, "experiments", "fine-tuning", "adapter-models", adapter_choice)



In [ ]:
# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    dtype=torch.float16,
    device_map="auto"
)

# Load adapters
model = PeftModel.from_pretrained(base_model, adapter_model_path)

# Merge adapters into base model
model = model.merge_and_unload()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Inference
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=500)



In [ ]:
#claim_text = "According to experts in the field, Inhabitat was founded by an architectural graduate student from a prominent private Ivy League research university in New York City in 2005, and by 2007, the site had a full-time salaried managing editor and around a dozen contributors who earned a nominal fee for their work."
#claim_text = "According to industry experts, Joey Graceffa has experience working with various digital platforms beyond video-sharing websites."
#claim_text = "According to a widely recognized authority on genealogy, it is stated that Jeff Bezos is Armenian."
claim_text = "Tomorrow is the same as today."

# your messages in HF format
messages = [{"role": "system", "content": "Evaluate this claim as 'SUPPORTS' or 'REFUTES'. Your answer should be a single string with the answer."}, {"role": "user", "content": f"{claim_text}"}]



# get the converted prompt
prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt_str = prompt_str.replace("<think>", "") # in case of deepseek


print(textwrap.fill(prompt_str, width=80))



In [ ]:
# Run inference
result = pipe(prompt_str)

#print(textwrap.fill(result[0]['generated_text'], width=80))
print(result[0]['generated_text'])

